In [1]:
import pandas as pd
import gc

In [2]:
df = pd.read_csv('../ex04/fines.csv')
df.head()

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2.0,3200.0,Ford,Focus,1989
1,E432XX77RUS,1.0,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1.0,2100.0,Ford,Focus,1984
3,X582HE161RUS,2.0,2000.0,Ford,Focus,2015
4,92918M178RUS,1.0,5700.0,Ford,Focus,2014


In [3]:
def loop_with_iloc(dataframe):
    result = []
    for i in range(0, len(dataframe)):
        fines = dataframe.iloc[i]['Fines']
        refund = dataframe.iloc[i]['Refund']
        year = dataframe.iloc[i]['Year']
        
        if pd.isna(refund) or refund == 0:
            result.append(None)
        else:
            result.append(fines / refund * year)
    return result

In [4]:
%%timeit
df['calc_iloc'] = loop_with_iloc(df)

73.5 ms ± 225 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


Using `iterrows()`

In [5]:
def loop_with_iterrows(dataframe):
    result = []
    for _, row in dataframe.iterrows():
        if pd.isna(row['Refund']) or row['Refund'] == 0:
            result.append(None)
        else:
            result.append(row['Fines'] / row['Refund'] * row['Year'])
    return result

In [6]:
%%timeit
df['calc_iterrows'] = loop_with_iterrows(df)

28 ms ± 411 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


Using `apply()` and a lambda function

In [7]:
%%timeit
df['calc_apply'] = df.apply(
    lambda row: None if pd.isna(row['Refund']) or row['Refund'] == 0
    else row['Fines'] / row['Refund'] * row['Year'],
    axis=1
)

9.76 ms ± 116 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Using pandas Series objects

In [8]:
%%timeit
refund = df['Refund']
fines = df['Fines']
year = df['Year']

df['calc_series'] = fines / refund * year


138 µs ± 2.43 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [9]:
%%timeit
refund = df['Refund'].values
fines = df['Fines'].values
year = df['Year'].values

df['calc_values'] = fines / refund * year

<magic-timeit>:5: RuntimeWarning: divide by zero encountered in divide
<magic-timeit>:5: RuntimeWarning: divide by zero encountered in divide


63.5 µs ± 518 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Get a row for a specific `CarNumber`

In [10]:
%%timeit
df[df['CarNumber'] == 'O136HO197RUS']

205 µs ± 8.94 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
indexed_df = df.set_index('CarNumber')
indexed_df.head()

,Refund,Fines,Make,Model,Year,calc_iloc,calc_iterrows,calc_apply,calc_series,calc_values
CarNumber,,,,,,,,,,
Y163O8161RUS,2.0,3200.0,Ford,Focus,1989,3182400.0,3182400.0,3182400.0,3182400.0,3182400.0
E432XX77RUS,1.0,6500.0,Toyota,Camry,1995,12967500.0,12967500.0,12967500.0,12967500.0,12967500.0
7184TT36RUS,1.0,2100.0,Ford,Focus,1984,4166400.0,4166400.0,4166400.0,4166400.0,4166400.0
X582HE161RUS,2.0,2000.0,Ford,Focus,2015,2015000.0,2015000.0,2015000.0,2015000.0,2015000.0
92918M178RUS,1.0,5700.0,Ford,Focus,2014,11479800.0,11479800.0,11479800.0,11479800.0,11479800.0


In [12]:
%%timeit
indexed_df.loc['O136HO197RUS']

46.9 µs ± 1.67 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Downcasting

Now we inspect memory usage and optimize numeric dtypes.

In [13]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CarNumber      930 non-null    object 
 1   Refund         928 non-null    float64
 2   Fines          930 non-null    float64
 3   Make           930 non-null    object 
 4   Model          919 non-null    object 
 5   Year           930 non-null    int64  
 6   calc_iloc      927 non-null    float64
 7   calc_iterrows  927 non-null    float64
 8   calc_apply     927 non-null    float64
 9   calc_series    928 non-null    float64
 10  calc_values    928 non-null    float64
dtypes: float64(7), int64(1), object(3)
memory usage: 232.9 KB


In [14]:
optimized_df = df.copy()
optimized_df

,CarNumber,Refund,Fines,Make,Model,Year,calc_iloc,calc_iterrows,calc_apply,calc_series,calc_values
0,Y163O8161RUS,2.0,3200.0,Ford,Focus,1989,3182400.0,3182400.0,3182400.0,3182400.0,3182400.0
1,E432XX77RUS,1.0,6500.0,Toyota,Camry,1995,12967500.0,12967500.0,12967500.0,12967500.0,12967500.0
2,7184TT36RUS,1.0,2100.0,Ford,Focus,1984,4166400.0,4166400.0,4166400.0,4166400.0,4166400.0
3,X582HE161RUS,2.0,2000.0,Ford,Focus,2015,2015000.0,2015000.0,2015000.0,2015000.0,2015000.0
4,92918M178RUS,1.0,5700.0,Ford,Focus,2014,11479800.0,11479800.0,11479800.0,11479800.0,11479800.0
...,...,...,...,...,...,...,...,...,...,...,...
925,A001AA199,NaN,1200.0,Lada,Vesta,2010,NaN,NaN,NaN,NaN,NaN
926,B002BB199,1.0,2300.0,BMW,X5,2012,4627600.0,4627600.0,4627600.0,4627600.0,4627600.0
927,C003CC199,0.0,750.0,Audi,A4,2015,NaN,NaN,NaN,inf,inf
928,D004DD199,NaN,980.0,Toyota,Camry,2018,NaN,NaN,NaN,NaN,NaN


Downcast float columns from `float64` to `float32`

In [15]:
float_columns = optimized_df.select_dtypes(include=['float64']).columns
optimized_df[float_columns] = optimized_df[float_columns].apply(pd.to_numeric, downcast='float')


In [16]:
int_columns = optimized_df.select_dtypes(include=['int64']).columns
int_columns

Index(['Year'], dtype='object')

In [17]:
optimized_df[int_columns] = optimized_df[int_columns].apply(pd.to_numeric, downcast='integer')

In [18]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CarNumber      930 non-null    object 
 1   Refund         928 non-null    float32
 2   Fines          930 non-null    float32
 3   Make           930 non-null    object 
 4   Model          919 non-null    object 
 5   Year           930 non-null    int16  
 6   calc_iloc      927 non-null    float64
 7   calc_iterrows  927 non-null    float64
 8   calc_apply     927 non-null    float64
 9   calc_series    928 non-null    float64
 10  calc_values    928 non-null    float64
dtypes: float32(2), float64(5), int16(1), object(3)
memory usage: 220.2 KB


In [19]:
object_columns = optimized_df.select_dtypes(include=['object']).columns
object_columns

Index(['CarNumber', 'Make', 'Model'], dtype='object')

In [20]:
optimized_df[object_columns] = optimized_df[object_columns].astype('category')

In [21]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   CarNumber      930 non-null    category
 1   Refund         928 non-null    float32 
 2   Fines          930 non-null    float32 
 3   Make           930 non-null    category
 4   Model          919 non-null    category
 5   Year           930 non-null    int16   
 6   calc_iloc      927 non-null    float64 
 7   calc_iterrows  927 non-null    float64 
 8   calc_apply     927 non-null    float64 
 9   calc_series    928 non-null    float64 
 10  calc_values    928 non-null    float64 
dtypes: category(3), float32(2), float64(5), int16(1)
memory usage: 103.1 KB


In [22]:
del df
gc.collect()
%reset_selective -f ^df$